# Analyse de la Complexité Théorique du Problème

## Rappel des différents problèmes :

![image.png](attachment:37efe983-e7d2-400a-b623-dc36f75e79ef.png)

![image.png](attachment:d735d47c-e1a3-46f3-b94d-71027ca051fa.png)

# 1. Pourquoi évaluer la difficulté de notre problème ?

Avant de développer des algorithmes pour optimiser les tournées de nos véhicules, il est essentiel de comprendre la nature mathématique du problème auquel nous sommes confrontés. En informatique théorique, tous les problèmes ne se valent pas, certains peuvent être résolus rapidement, quelle que soit leur taille, tandis que d'autres voient leur temps de résolution exploser dès que l'on ajoute quelques variables.

Dans le cadre de cet appel à projets de l'ADEME, nous devons traiter des réseaux de transport à l'échelle de territoires entiers. Notre modèle doit gérer des contraintes temporelles strictes (les fenêtres de livraison de chaque client) et s'adapter dynamiquement aux restrictions du réseau routier (zones à faibles émissions, routes barrées). L'objectif de cette section est de démontrer formellement la complexité de ce problème pour justifier nos futurs choix algorithmiques.

## 2. Énoncé du Théorème

### A. Le Problème de Décision

Le problème d'optimisation de tournée avec fenêtres temporelles et restrictions de passage (que nous nommerons TSPTW-Restreint, pour Traveling Salesperson Problem with Time Windows) est un problème NP-difficile.

En termes simples, qualifier un problème de "NP-difficile" signifie qu'il n'existe à ce jour aucun algorithme connu capable de trouver la solution absolument parfaite en un temps d'exécution raisonnable (polynomial) pour toutes les instances du problème.
Transformons notre problème d'optimisation (trouver le trajet le plus court) en un problème de décision.La question que l'on pose à l'algorithme devient :
- "Étant donné notre réseau de villes (incluant des routes potentiellement interdites ou restreintes), notre véhicule unique, et nos contraintes de créneaux horaires pour chaque client, existe-t-il une tournée dont le coût total (en temps ou en distance) est inférieur ou égal à une valeur cible $C$ ?"

### B. L'Algorithme de Vérification

```
Fonction VerifierSolution(Route, MatriceCouts, FenetresTemps, AretesInterdites, Cible_C):
    CoutTotal = 0
    TempsActuel = 0
    VillesLivrees = Liste vide
    NbVilles = longueur(Route) - 2 // On exclut le dépôt de départ/arrivée

    // Parcours de la route étape par étape
    Pour chaque etape i de 0 à NbVilles:
        VilleActuelle = Route[i]
        VilleSuivante = Route[i+1]

        // Vérification 1 : Restriction de passage (ZFE ou route barrée)
        Si (VilleActuelle, VilleSuivante) est dans AretesInterdites:
            Retourner FAUX (Emprunt d'une route interdite)

        // Trajet vers la ville suivante
        TempsDeTrajet = MatriceCouts[VilleActuelle][VilleSuivante]
        TempsActuel = TempsActuel + TempsDeTrajet
        CoutTotal = CoutTotal + TempsDeTrajet

        // Vérification 2 : Fenêtre temporelle de la ville suivante
        HeureOuverture = FenetresTemps[VilleSuivante].Debut
        HeureFermeture = FenetresTemps[VilleSuivante].Fin

        Si TempsActuel > HeureFermeture:
            Retourner FAUX (Livraison en retard, client fermé)

        Si TempsActuel < HeureOuverture:
            // Le livreur est en avance, il doit attendre l'ouverture
            TempsActuel = HeureOuverture

        Ajouter VilleSuivante à VillesLivrees

    // Vérification 3 : Toutes les villes ont-elles été livrées ?
    Si VillesLivrees contient des doublons ou s'il manque des villes:
        Retourner FAUX (Erreur de livraison)

    // Vérification 4 : Le coût total respecte-t-il la cible ?
    Si CoutTotal > Cible_C:
        Retourner FAUX (Objectif non atteint)

    Retourner VRAI (Solution valide)
```

À la lecture de notre algorithme de vérification, on observe une unique boucle principale qui parcourt les étapes de la route du véhicule.Puisque dans une solution valide, le véhicule visite exactement une fois chaque client, cette boucle s'exécutera exactement $n$ fois (où $n$ est le nombre total de clients à livrer sur le territoire).

À l'intérieur de cette boucle, l'algorithme n'effectue que des opérations simples et instantanées :
- Des additions (pour cumuler le temps de trajet et le coût total).
- Des comparaisons conditionnelles (pour vérifier si l'heure d'arrivée respecte les fenêtres temporelles et si la route est autorisée).
- L'ajout de la ville visitée dans une structure de données (comme un set en Python) pour s'assurer par la suite qu'il n'y a aucun doublon.

En informatique, la lecture d'une variable ou l'accès à une table de hachage s'exécute en un temps constant, noté $\mathcal{O}(1)$. Puisque notre algorithme exécute un nombre d'opérations constantes de manière strictement proportionnelle au nombre de villes ($n \times \mathcal{O}(1)$), son temps d'exécution global grandit de façon purement linéaire.La complexité de notre algorithme de vérification est donc bien en $\mathcal{O}(n)$.
Cette capacité à vérifier une solution en temps polynomial est la condition stricte qui prouve que notre problème appartient formellement à la classe NP.

_

Par conséquent, puisque notre programme effectue un nombre limité d'opérations instantanées exactement $n$ fois, le temps de réponse total de notre algorithme de vérification grandira de manière purement proportionnelle au nombre de clients. C'est la définition même d'une complexité linéaire en $O(n)$.

## C. Preuve de NP-Complétude du Problème du Voyageur de Commerce (TSP)

Avant d'analyser notre problème spécifique de livraison, nous devons prouver la complexité de son socle fondamental : le Problème du Voyageur de Commerce (TSP) sur un graphe incomplet.

Pour prouver qu'un problème de décision est **NP-Complet**, la démarche mathématique exige de démontrer deux choses :
1. Qu'il appartient à la classe **NP** (on peut vérifier une solution rapidement).
2. Qu'il est **NP-difficile** (on peut y réduire un problème NP-Complet déjà connu en un temps polynomial).

### Étape 1 : Le TSP appartient à la classe NP

**Problème de décision du TSP :** Étant donné un graphe $G=(V,E)$ pondéré et un entier $k$, existe-t-il un cycle simple passant par tous les sommets de $V$ exactement une fois, avec un coût total inférieur ou égal à $k$ ?

**Algorithme de vérification :**
Si l'on nous donne un cycle candidat (une liste de sommets), nous pouvons vérifier s'il est valide en :
1. Comptant le nombre de sommets visités pour s'assurer qu'ils y sont tous, sans doublons.
2. Vérifiant que chaque arête empruntée existe bien dans l'ensemble $E$.
3. Faisant la somme des poids de ces arêtes et en vérifiant qu'elle est $\le k$.

Ce parcours ne nécessite qu'une seule boucle sur les $n$ sommets du graphe. Le temps de vérification est donc de complexité linéaire $\mathcal{O}(n)$, ce qui est un temps polynomial. Le TSP appartient donc formellement à la classe **NP**.

### Étape 2 : Le TSP est NP-difficile (Preuve par réduction)

Pour prouver la NP-difficulté, nous allons réduire un problème célèbre déjà prouvé comme étant NP-Complet : le **Problème du Cycle Hamiltonien (HC)**, vers notre problème TSP.

**Rappel du Cycle Hamiltonien (HC) :** Étant donné un graphe non pondéré $G=(V,E)$, existe-t-il un cycle simple passant par chaque sommet exactement une fois ?

**La Transformation (Réduction) :**
À partir d'une instance quelconque du problème HC (un graphe $G=(V,E)$ de $n$ sommets), construisons une instance du TSP :
1. Prenons le même ensemble de sommets $V$.
2. Créons un graphe complet $G'=(V,E')$ où toutes les paires de sommets sont reliées par une arête.
3. Attribuons les coûts suivants aux arêtes de $G'$ :
   - Si l'arête $(u,v)$ existe dans le graphe d'origine $G$, son coût est de $1$.
   - Si l'arête $(u,v)$ n'existe pas dans le graphe d'origine $G$, son coût est de $2$.
4. Fixons notre coût cible pour le TSP à $k = n$.

Cette transformation se fait en parcourant les paires de sommets, soit en un temps $\mathcal{O}(n^2)$, ce qui est bien polynomial.

### Étape 3 : Équivalence des solutions

Prouvons maintenant que la solution au problème HC existe **si et seulement si** la solution au problème TSP existe :

- **Sens $\Rightarrow$ :** S'il existe un cycle Hamiltonien dans le graphe $G$, ce cycle comprend exactement $n$ arêtes. Dans notre nouveau graphe $G'$, toutes ces arêtes ont un poids de $1$. Le coût total du cycle dans $G'$ est donc de $n \times 1 = n$. La condition du TSP (coût $\le k$ avec $k=n$) est remplie.
- **Sens $\Leftarrow$ :** S'il existe une solution au TSP dans $G'$ avec un coût total $\le n$, sachant que le cycle doit obligatoirement comporter $n$ arêtes et que le coût minimum d'une arête est $1$, la seule façon d'obtenir un coût de $n$ est de n'emprunter que des arêtes de poids $1$. Par définition de notre réduction, ces arêtes de poids $1$ appartiennent toutes au graphe d'origine $G$. Donc, ce cycle est un cycle Hamiltonien valide pour $G$.

**Conclusion :** Puisque le Problème du Cycle Hamiltonien se réduit polynomialement au Problème du Voyageur de Commerce (HC $\le_p$ TSP) et que le HC est NP-Complet, **le problème de décision du TSP est  NP-Complet.**

D'une part, grâce à notre algorithme de vérification, nous avons prouvé qu'il est possible de vérifier la validité d'une solution en temps linéaire. Cette rapidité de vérification est la condition stricte pour qu'un problème décisionnel appartienne à la classe NP.

D'autre part, notre problème d'optimisation est intrinsèquement NP-difficile. Comme nous l'avons démontré par la méthode de restriction, la recherche de l'itinéraire le plus court englobe le célèbre Problème du Voyageur de Commerce (TSP). L'ajout de nos deux contraintes de terrain les fenêtres temporelles imposées par les clients et les restrictions de passage (comme les zones à faibles émissions) vient restreindre drastiquement l'espace des solutions réalisables. Trouver le bon trajet n'est plus seulement une question de distance géographique, c'est devenu un véritable casse-tête de synchronisation horaire.

Pour résumer, notre modèle logistique combine une difficulté spatiale majeure (le Problème du Voyageur de Commerce) avec des exigences temporelles strictes. Le TSP de base étant mathématiquement classé comme NP-difficile, cette version sur-contrainte de notre modèle de transport reste par extension tout aussi complexe à résoudre de manière exacte.

En pratique, cette complexité se traduit par ce qu'on appelle une "explosion combinatoire". Le nombre de tournées possibles se multiplie de façon vertigineuse à chaque fois que l'on ajoute un client. Si un ordinateur peut facilement tester toutes les options pour 10 villes, la tâche devient impossible pour une échelle territoriale réaliste de 50 points de livraison. Il y a un tel nombre de combinaisons que même les machines les plus puissantes mettraient des années à toutes les évaluer pour garantir un trajet absolument parfait.

Puisqu'une méthode de calcul exacte (la "force brute") prendrait beaucoup trop de temps, nous allons utiliser des algorithmes d'optimisation approchée (les heuristiques et métaheuristiques). L'objectif n'est plus de chercher la perfection absolue, mais d'explorer intelligemment les possibilités pour fournir une solution très performante. Cela nous permettra d'optimiser les tournées, de réduire l'impact carbone et de respecter des temps de calcul de quelques secondes, comme l'exige la réalité du terrain chez CesiCDP.